In [222]:
!#pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [223]:
# Google Auth Imports
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

#Core Plumbing Imports
import os
from dotenv import load_dotenv
import json
import base64

#Diplay imports
from pprint import pprint
from IPython.display import Markdown, display

#Tool Imports
from ddgs import DDGS
import trafilatura
import io

#AI Library Imports
from google import genai
from agents import Agent, Runner, function_tool, trace

In [224]:
load_dotenv()

True

### Step 0: Setup and Configuration

In [225]:
SCOPES = ["https://www.googleapis.com/auth/drive.file"]

if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
else:
    flow = InstalledAppFlow.from_client_secrets_file("client_secret.json", SCOPES)
    creds = flow.run_local_server(port=0)
    with open("token.json", "w") as f:
        f.write(creds.to_json())

In [226]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if OPENAI_API_KEY.startswith("sk-proj"):
    print('API Key is ready')
else: 
    print('The key has an issue')

API Key is ready


In [227]:
MODEL = "gpt-4.1-nano"
gemini_client = genai.Client()

### Step 1: Define Tools

In [228]:
@function_tool
def search_web(query: str):
    """Search the web using Duck Duck Go. Returns 5 results"""
    ddgs = DDGS()
    results = ddgs.text(query,max_results=5)
    print(f" \u2705 Got results")
    return json.dumps(results, indent=2)

In [229]:
@function_tool
def get_url(url: str):
    """Fetch the content of a URL using Trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 got text: {len(text)} chars")
            return text
    print(f" \u274c Failed to fetch or extract text.")
    return f"Could not extract text from {url}. Try a different source."

In [230]:
def generate_image(prompt: str) -> str:
    # Step 1: State the prompt
    print(f"   Generate image base on this prompt: {prompt[:60]}...")
    # Step 2: Call for the image to be generated
    interaction = gemini_client.interactions.create(
        model="gemini-3.1-flash-image",
        input=prompt,
        response_format=[{
            "type": "image", 
            "mime_type": "image/jpeg",
            "aspect_ratio": "16:9",
            "image_size": "2K"
        }],
    )
    #Step 3: Return the image bytes
    return base64.b64decode(interaction.output_image.data)


In [231]:
@function_tool
def send_image_to_cloud(prompt: str, image_name: str):
    """Use Gemini to generate an image. The prompt should be a detailed visual description."""

    #Step 1: Generate the image and save the returned value
    image_data = generate_image(prompt)

    #Step 2: Set up the connection to Google Drive
    drive_service = build("drive", "v3", credentials=creds)

    #Step 3: Set up the file to information to be uploaded
    file_metadata = {"name": f"{image_name}.png"}
    media = MediaIoBaseUpload(io.BytesIO(image_data), mimetype="image/png", resumable=True)

    #Step 4: Upload the file and get an identifier
    uploaded_file = drive_service.files().create(
    body=file_metadata,
    media_body=media,
    fields="id, webViewLink"
    ).execute()
    file_id = uploaded_file["id"]

    #Step 5: Set Read Permissions on the file
    drive_service.permissions().create(
    fileId=file_id,
    body={"type": "anyone", "role": "reader"},
    ).execute()

    #Step 6: 
    result = drive_service.files().get(fileId=file_id, fields="webViewLink").execute()
    print("View link:", result["webViewLink"])
    return result["webViewLink"]
    

### Step 2: The Agents

#### Research Agent

In [232]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search, you MUST first explain your reasoning:
- Which URLs look most relevant and why
- Which ones you will fetch and why
- Which ones you are skipping and why
Only AFTER writing out your reasoning should you call fetch_url.***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief.

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent("Research Agent", instructions=RESEARCH_AGENT_PROMPT,model=MODEL, tools=[search_web, get_url])

#### Image Generating Agent

In [ ]:
IMAGE_GENERATION_AGENT_PROMPT = """
    You create images using Gemini. To do this, you write 
    image generation prompts which you send to the send_image_to_cloud
    tool you have access to. You also provide that tool a name for the image
    that is generated.

    !IMPORTANT: Your output should be the Google Drive URL that send_image_to_cloud provides you. Only call the send_image_to_cloud 1 time.
    An effective prompt for Gemini includes the following elements:

    1. The description of a style for the image (such as but not restricted to natural, stylistic, or cartoon).
    2. A detailed description of the image itself. A description should use words that could be verified by looking at the image objectively. Avoid subjective descriptions that could not be verified objectively.
    3. A maximum of 200 words.
    4. Requests for an image only, with no text, logos, words, or real human faces incldued in the image.
    5. No icon dumps or collages.
    6. Requests a single image, not multiple
    7. Is specific about lighting, composition, and mood
"""
image_gen_agent = Agent("Image Generation Agent", instructions=IMAGE_GENERATION_AGENT_PROMPT,model=MODEL, tools=[send_image_to_cloud])

#### Set Agents as Tools

In [234]:
research_tool = research_agent.as_tool(
    tool_name="research_agent",
    tool_description="Research a topic and return a brief with key facts, statistics, themes, and source URLs. Pass the topic as an input.",
    max_turns=20
)
image_gen_tool = image_gen_agent.as_tool(
    tool_name="image_gen_agent",
    tool_description="Generate a hero image for an article based on a topic and content summary. Supply the topic and content summary",
    max_turns=4
)

#### Orchestrator Agent

In [ ]:
ORCHESTRATOR_AGENT_PROMPT = """
You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and agents to produce a high-quality article.
Never do the work yourself. Always use tools or agents.
Your tools and agents are specialists; you are the manager.

## STEP 1 — RESEARCH (exactly two calls, no more, no less)
Call research_agent twice. Do NOT vary the inputs randomly. Vary them along
a deliberate axis so the two briefs give you a real choice:
  - Brief A: angled toward evidence, mechanisms, claims, data, and who disputes them.
  - Brief B: angled toward people, events, consequences, and lived experience.

## STEP 2 — SELECT ONE BRIEF
Pick ONE brief. Do not merge them. Do not add to them.
"Best" means: most specific, best sourced, and richest in usable material —
NOT the one that is longest or most agreeable.
Note which of these the winning brief actually contains, because Step 3 depends on it:
  - named_people: are there real, named human actors with actions and stakes?
  - contested: are there claims that credible parties actively dispute?
  - decision: does the reader face a choice they can act on?
  - novelty: is there a genuinely surprising, well-supported finding?
  - sensitivity: does the topic involve death, illness, violence, disaster,
    crime victims, or discrimination against a group?

## STEP 3 — ROUTE TO ONE WRITER
Apply in order. Stop at the first decision.

HARD BLOCKS (eliminate before scoring):
  - sensitivity = true            -> eliminate Humorist. Always. No exceptions.
  - named_people = false          -> eliminate Storyteller (it will invent people).
  - contested = false             -> eliminate Skeptic (it becomes contrarian noise
                                     when attacking well-settled material).
  - decision = false              -> eliminate Advisor (it will invent advice).
  - novelty = false               -> eliminate Journalist (it will overstate a
                                     mundane finding to manufacture a lede).
  - topic is a live political/social controversy -> eliminate Journalist
    (it takes a stance) and Skeptic. Prefer Educator or Interviewer.

SELECT by dominant reader need among survivors:
  UNDERSTAND — reader knows nothing and needs to be built up from zero
      -> Educator, unless the topic decomposes cleanly into discrete
         standalone questions, then -> Interviewer
  DECIDE    — reader has agency and a real choice in front of them -> Advisor
  DOUBT     — the subject is over-claimed, hyped, or thinly evidenced -> Skeptic
  DISCOVER  — there is a strong, well-supported, surprising finding
              with stakes and a defensible position -> Journalist
  FEEL      — the material's power is human and specific, with real people,
              scenes, and a timeline -> Storyteller
              ... if the resonance is abstract, sensory, or emotional rather
              than tied to specific people or events -> Poet
  ENJOY     — the absurdity is genuinely in the subject itself and the
              stakes are low -> Humorist

TIE-BREAKS for the pairs that collide most:
  Journalist vs Skeptic  : evidence is strong -> Journalist. Evidence is weak
                           or the claims outrun the proof -> Skeptic.
  Educator vs Interviewer: needs cumulative scaffolding -> Educator.
                           Splits into independent questions -> Interviewer.
  Advisor vs Journalist  : reader can act -> Advisor. Reader is a spectator
                           -> Journalist.
  Storyteller vs Poet    : real named people doing things -> Storyteller.
                           No people, or the subject is a concept -> Poet.
  Any tie unresolved     : Educator. It is the safe default and fails gracefully.

## STEP 4 — IMAGE (exactly one call)
Only after the writer is selected. Call image_gen_tool ONCE.
Supply the topic and content summary from the brief, PLUS the visual register
of the selected writer:
  Journalist  -> documentary photojournalism, high contrast, real-world
  Storyteller -> cinematic, character-centered, warm, shallow depth of field
  Skeptic     -> clinical, austere, diagrammatic, cold light
  Educator    -> clean illustrated diagram, labeled, bright, uncluttered
  Advisor     -> minimal editorial graphic, structured, restrained palette
  Humorist    -> playful, absurd, exaggerated scale, saturated
  Interviewer -> portrait-framed, studio lighting, conversational staging
  Poet        -> atmospheric, painterly, abstract, textural

## STEP 5 — HANDOFF
Hand off to the SELECTED writer agent only. Include:
  - the full selected research brief, unmodified
  - the image URL
  - a 2-3 sentence assignment note: the angle, what to emphasize, what to avoid
  - this constraint, verbatim:
    "Every factual claim, quote, statistic, and attributed statement must come
     from the brief. You may invent framing, structure, voice, scene-setting,
     and connective language. You may NOT invent facts, sources, named
     individuals, or statements attributed to real people."

Before handing off, emit:
  {
    "brief_selected": "A" | "B",
    "brief_signals": {"named_people": bool, "contested": bool, "decision": bool,
                      "novelty": bool, "sensitivity": bool},
    "writer_selected": "",
    "runner_up": "",
    "why_not_runner_up": "",
    "assignment_note": ""
  }

IMPORTANT: When returning the image URL, copy it EXACTLY character by
character. Do not modify, shorten, or add additional characters.
"""
orchestrator_agent = Agent("Orchestrator Agent", instructions=ORCHESTRATOR_AGENT_PROMPT, model="o4-mini", tools=[research_tool, image_gen_tool])

#### Interviewer Interviewer

In [ ]:
INTERVIEWER_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

interviewer_agent = Agent("Interviewer Agent", instructions=INTERVIEWER_AGENT_PROMPT,model=MODEL)

#### Humorist Agent

In [ ]:
HUMORIST_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

humorist_agent = Agent("Humorist Agent", instructions=HUMORIST_AGENT_PROMPT,model=MODEL)

#### Poet Agent

In [ ]:
POET_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

poet_agent = Agent("Poet Agent", instructions=POET_AGENT_PROMPT,model=MODEL)

#### Advisor Agent

In [ ]:
ADVISOR_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

advisor_agent = Agent("Advisor Agent", instructions=ADVISOR_AGENT_PROMPT,model=MODEL)

#### Skeptic Agent

In [ ]:
SKEPTIC_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

skeptic_agent = Agent("Skeptic Agent", instructions=SKEPTIC_AGENT_PROMPT,model=MODEL)

#### Educator Agent

In [ ]:
EDUCATOR_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

educator_agent = Agent("Educator Agent", instructions=EDUCATOR_AGENT_PROMPT,model=MODEL)

#### Storyteller Agent

In [ ]:
STORYTELLER_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

storyteller_agent = Agent("Storyteller Agent", instructions=STORYTELLER_AGENT_PROMPT,model=MODEL)

#### Journalist Agent

In [236]:
JOURNALIST_AGENT_PROMPT= """
You are an investigative journalist. You write articles with a clear point of view, valid data, and a journalistic style reflective of the typical article style, with the most important, newsworthy information at the beginning of the article, and less important, supporting information later in the article.
You will be provided with a research brief on a topic that you can use. The brief will include a summary of available information and a set of links for sources. 

Your style is similar to that of Mehdi Hasan: sharp, pointed, and uncompromising.
You challenge assumptions and ask hard questions throughout 
You take a clear stance — you have an opinion and you're not afraid to share it 
You quote sources and reference specific data points 
You structure like a news feature: hook, context, evidence, tension, conclusion 
You aim for 800-1200 words 

Do NOT make up facts that you have not verified. 
Do NOT present both sides of an argument. 
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.

"""

journalist_agent = Agent("Journalist Agent", instructions=JOURNALIST_AGENT_PROMPT,model=MODEL)

##### Update the Orchrestrator Agent

In [237]:
orchestrator_agent.handoffs = [journalist_agent]

In [238]:
# with trace("Journalist Writer", group_id="Learning AI Engineering"):
#     result = await Runner.run(
#         journalist_agent,
#         input = f"The impact of bananas on the modern economy.",
#         max_turns=30
#     )
# print(result.final_output)

### Step 3: Run the Orchestrator

In [239]:
with trace("Article Writer w/ Handoff", group_id="Learning AI Engineering"):
    result = await Runner.run(
        orchestrator_agent,
        input = f"How will the rise of China impact global culture in the next 30 years?.",
        max_turns=30
    )

 ✅ Got results
 ✅ Got results
 ✅ got text: 321 chars
 ✅ got text: 6470 chars
 ✅ got text: 5897 chars
 ✅ Got results
 ✅ Got results
 ✅ Got results
 ❌ Failed to fetch or extract text.
 ✅ got text: 3093 chars
 ❌ Failed to fetch or extract text.
   Generate image base on this prompt: A futuristic, highly detailed digital painting in a modernis...   Generate image base on this prompt: A stylized, semi-abstract image illustrating China's expandi...
   Generate image base on this prompt: A highly detailed, photorealistic image representing the str...

   Generate image base on this prompt: A sleek, digital art-style representation of the multipolar ...
View link: https://drive.google.com/file/d/1o_Rh0GWDsevnvxO7lyCbSQKEQ56i_gli/view?usp=drivesdk
View link: https://drive.google.com/file/d/1ILAWxKnyyk14cUT_Fco7EF2_gHgHDVWS/view?usp=drivesdk
View link: https://drive.google.com/file/d/11lqH1_wRapwYzudz8FiFx8NLQBw1snrE/view?usp=drivesdk
View link: https://drive.google.com/file/d/18_T3OS-Q87ocJ5ZY9

In [240]:
# print(f"Agent {result.last_agent.name}")
# print(f"---")
# display(Markdown(result.final_output))
# pprint(result.final_output)